# Préparation des données DPO — MediQAl (mcqm)

Construction d'un jeu de paires préférentielles **en français** (`prompt` / `chosen` / `rejected`) pour l'alignement DPO, à partir des QCM à réponses multiples de MediQAl (configuration `mcqm`, licence CC-BY-4.0). Cette configuration n'a pas servi au SFT, qui utilise `mcqu`.

- **chosen** : toutes les bonnes réponses, d'après le corrigé officiel.
- **rejected** : la même liste avec **une erreur**, construite à partir des distracteurs (mauvaises options) rédigés par des enseignants en médecine. L'erreur est donc plausible, et le corrigé garantit qu'elle est fausse.
- **Qualité plutôt que quantité** : une seule paire par question, uniquement les questions rattachées à un cas clinique, sans option ambiguë.
- **Sortie** : `data/anonymized/dpo_mediqal_mcqm_{train,validation,test}.anonymized.jsonl`, au format conversationnel attendu par `trl.DPOTrainer`.

# Import

In [1]:
import json
import re

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset

from projet_14.rgpd import run

SEED = 42
LETTERS = list("abcde")

# Options trop courtes (« 1 », « Oui »...) : peu informatives sans la question complète
MIN_OPTION_CHARS = 3

# Taille maximale des jeux après filtrage (le test est gardé entier)
N_TRAIN = 2000
N_VALIDATION = 300

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Get hugging face dataset

In [2]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("ANR-MALADES/MediQAl", "mcqm")
splits = {name: ds[name].to_pandas() for name in ds}
df_train = splits["train"]

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


# Analyse dataset

In [3]:
df_train.head()

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type
0,11399,Plusieurs jours après un contage avec un enfan...,Quelle(s) proposition(s) peut (peuvent) s'appl...,Transmission manuportée,La phase de contagiosité dure 8 jours,Maladie strictement humaine,Confère une immunité,Transmission indirecte possible,"C,D",QCM,Infectious Diseases,Understanding
1,17928,NaN,Quel(s) germe(s) est (sont) responsable(s) de ...,Gardnerella Vaginalis,Trichomonas,Tréponèmes,Candida albicans,Strepto B,"A,B",QCM,Gynecology and Obstetrics,Understanding
2,17520,"Une jeune femme de 26 ans, d'origine française...","En cours d'hospitalisation, laquelle ou lesque...",Réduction de l'apport en protéines animales,Association d'une sulfonylurée à l'insuline,Conservation du même type d'insuline mais augm...,Education à l'autosurveillance glycémique,"Adaptation d'un nouveau schéma thérapeutique, ...","A,D,E",QCM,Endocrinology and Metabolism,Reasoning
3,10875,NaN,Quelles sont parmi les bactéries suivantes res...,Staphylococcus aureus,Salmonella enteritidis,Shigella sonnei,Vibrio cholerae,Campylobacter fetus,"B,C,E",QCM,Pediatrics,Understanding
4,14691,NaN,"Parmi les 5 propositions suivantes, quelle est...",Il est plus souvent d'origine maternelle,La majorité des formes sont inapparentes,Une primo-infection génitale maternelle durant...,L'accouchement par césarienne est indiqué lors...,Aucune chimiothérapie antivirale n'est active,"A,C,D",QCM,Pediatrics,Understanding


In [4]:
# Nombres de lignes dans les différentes parties du dataset
for name, df in splits.items():
    print(f"{name}: {len(df)} rows, dont {df['clinical_case'].notna().sum()} avec un cas clinique")

train: 5767 rows, dont 2185 avec un cas clinique
validation: 1466 rows, dont 547 avec un cas clinique
test: 3384 rows, dont 1342 avec un cas clinique


In [5]:
# Nombre de bonnes réponses par question
n_correct = df_train["correct_answers"].str.split(",").map(len)
n_correct.value_counts().sort_index().rename_axis("bonnes réponses").to_frame("questions")

,questions
bonnes réponses,
2,2054
3,2234
4,1154
5,325


In [6]:
# Valeurs distinctes de "question_type" et "medical_subject"
print(df_train["question_type"].value_counts(), end="\n\n")
print(df_train["medical_subject"].value_counts())

question_type
Understanding    4320
Reasoning        1447
Name: count, dtype: int64

medical_subject
Cardiology                          395
Hepato-Gastroenterology             395
Pulmonology                         341
Psychiatry                          329
Nephro-Urology                      328
Endocrinology and Metabolism        300
Infectious Diseases                 291
Gynecology and Obstetrics           290
Rheumatology                        289
Hematology                          281
Neurology                           278
Dermatology                         270
Pediatrics                          249
Otorhinolaryngology (ENT)           220
Ophthalmology                       196
Forensic Medicine and Toxicology    172
Immunology                          158
Pharmacology                        147
Biochemistry                        142
Pathological Anatomy                141
Orthopedics                         127
Microbiology                        100
Epidemiology       

In [7]:
# Longueur des options (en caractères)
options = pd.concat([df_train[f"answer_{letter}"] for letter in LETTERS])
options.str.len().describe()

count    28835.000000
mean        33.046818
std         21.357796
min          1.000000
25%         18.000000
50%         28.000000
75%         43.000000
max        166.000000
dtype: float64

# Nettoyage

In [8]:
def clean_text(text: str) -> str:
    """Retire les espaces en début / fin de ligne et les lignes vides en trop."""
    text = "\n".join(line.strip() for line in text.splitlines())
    return re.sub(r"\n{3,}", "\n\n", text).strip()


TEXT_COLUMNS = ["clinical_case", "question"] + [f"answer_{letter}" for letter in LETTERS]
for name, df in splits.items():
    splits[name] = df.assign(**{col: df[col].map(clean_text, na_action="ignore") for col in TEXT_COLUMNS})

In [9]:
# Comme pour le SFT : uniquement les questions rattachées à un cas clinique (contexte patient, proche du triage)
for name, df in splits.items():
    n_before = len(df)
    df = df.dropna(subset=["clinical_case", "question"])
    splits[name] = df.drop_duplicates(subset=["clinical_case", "question"])
    print(f"{name}: {n_before} rows, {len(df)} avec un cas clinique, {len(splits[name])} après dédoublonnage")

train: 5767 rows, 2185 avec un cas clinique, 2182 après dédoublonnage
validation: 1466 rows, 547 avec un cas clinique, 547 après dédoublonnage
test: 3384 rows, 1342 avec un cas clinique, 1341 après dédoublonnage


In [10]:
# Options « méta » qui renvoient aux autres options : sans la liste complète, elles n'ont pas de sens
META_OPTION = re.compile(
    r"\b(?:propositions?|réponses?)\s+(?:ci-dessus|précédentes?|[A-E])\b"
    r"|\b(?:aucune|toutes)\s+(?:les|des|de\s+ces)\s+(?:propositions|réponses)",
    re.IGNORECASE,
)
option_columns = [f"answer_{letter}" for letter in LETTERS]

for name, df in splits.items():
    meta = df[option_columns].apply(lambda col: col.str.contains(META_OPTION)).any(axis=1)
    too_short = df[option_columns].apply(lambda col: col.str.len() < MIN_OPTION_CHARS).any(axis=1)
    splits[name] = df[~meta & ~too_short]
    print(f"{name}: {meta.sum()} questions avec option méta, {too_short.sum()} avec option trop courte, {len(splits[name])} rows")

train: 16 questions avec option méta, 4 avec option trop courte, 2162 rows
validation: 1 questions avec option méta, 0 avec option trop courte, 546 rows
test: 14 questions avec option méta, 7 avec option trop courte, 1320 rows


# Construction des paires DPO (chosen / rejected)

La réponse **chosen** est la liste des bonnes réponses. La réponse **rejected** contient une seule erreur, d'un des trois types suivants (tiré au hasard parmi ceux possibles) :

| Type | Construction | Erreur clinique simulée |
|---|---|---|
| `omission` | une bonne réponse retirée | oublier un élément (un signe de gravité, un examen) |
| `ajout` | un distracteur ajouté | retenir un élément faux |
| `substitution` | une bonne réponse remplacée par un distracteur | se tromper d'élément, à longueur égale |

Les trois types se compensent : `omission` raccourcit la réponse, `ajout` l'allonge, `substitution` garde le même nombre d'éléments. Le modèle ne peut donc pas apprendre « plus long = meilleur » (ni l'inverse). Les éléments sont toujours affichés dans l'ordre des options (A à E) : la position ne trahit pas l'erreur.

In [11]:
rng = np.random.default_rng(SEED)


def render(letters, row) -> str:
    """Liste à puces des options, dans l'ordre A à E (même format que les réponses multiples du SFT)."""
    return "\n".join(f"- {row[f'answer_{letter}']}" for letter in sorted(letters))


def make_pair(row) -> pd.Series:
    correct = {letter.strip().lower() for letter in row["correct_answers"].split(",")}
    wrong = sorted(set(LETTERS) - correct)
    removed = str(rng.choice(sorted(correct)))
    added = str(rng.choice(wrong)) if wrong else None

    candidates = {
        "omission": correct - {removed},
        "ajout": correct | {added},
        "substitution": (correct - {removed}) | {added},
    }
    if not wrong:  # toutes les options sont justes : seule l'omission est possible
        candidates = {"omission": candidates["omission"]}
    rejected_type = str(rng.choice(list(candidates)))
    return pd.Series(
        {
            "chosen": render(correct, row),
            "rejected": render(candidates[rejected_type], row),
            "rejected_type": rejected_type,
        }
    )


for name, df in splits.items():
    df = pd.concat([df, df.apply(make_pair, axis=1)], axis=1)
    splits[name] = df[df["chosen"] != df["rejected"]]

pd.concat({name: df["rejected_type"].value_counts() for name, df in splits.items()}, axis=1)

,train,validation,test
rejected_type,,,
omission,761,197,479
ajout,703,185,452
substitution,698,164,389


In [12]:
# Un exemple de chaque type
for rejected_type in ["omission", "ajout", "substitution"]:
    example = splits["train"][splits["train"]["rejected_type"] == rejected_type].iloc[0]
    print(f"===== {rejected_type}\nQUESTION : {example['question']}\n")
    print(f"CHOSEN :\n{example['chosen']}\n\nREJECTED :\n{example['rejected']}\n")

===== omission
QUESTION : Chez une personne de 70 ans comment prévenir une telle hyponatrémie ?

CHOSEN :
- Ne pas augmenter les apports hydriques quand on prescrit un diurétique thiazidique
- Ne pas prescrire un régime sans sel strict chez les gens âgés

REJECTED :
- Ne pas prescrire un régime sans sel strict chez les gens âgés

===== ajout
QUESTION : Quelle(s) proposition(s) peut (peuvent) s'appliquer à l'épidémiologie de la maladie ?

CHOSEN :
- Maladie strictement humaine
- Confère une immunité

REJECTED :
- Maladie strictement humaine
- Confère une immunité
- Transmission indirecte possible

===== substitution
QUESTION : En cours d'hospitalisation, laquelle ou lesquelles des attitudes suivantes allez-vous retenir ?

CHOSEN :
- Réduction de l'apport en protéines animales
- Education à l'autosurveillance glycémique
- Adaptation d'un nouveau schéma thérapeutique, p.ex. la pompe à infusion sous-cutanée ou l'association matin et soir d'insulines ordinaire et semi-lente

REJECTED :
- Ré

In [13]:
# Biais de longueur : part des paires où la chosen est la plus longue
for name, df in splits.items():
    longer = (df["chosen"].str.len() > df["rejected"].str.len()).mean()
    print(f"{name}: chosen plus longue dans {longer:.0%} des paires")

train: chosen plus longue dans 52% des paires
validation: chosen plus longue dans 51% des paires
test: chosen plus longue dans 52% des paires


# Échantillonnage

On garde le découpage d'origine de MediQAl (train / validation / test). Qualité plutôt que quantité : on plafonne le train à `N_TRAIN` paires et la validation à `N_VALIDATION`. Le test est gardé entier.

In [14]:
splits["train"] = splits["train"].sample(n=min(N_TRAIN, len(splits["train"])), random_state=SEED)
splits["validation"] = splits["validation"].sample(n=min(N_VALIDATION, len(splits["validation"])), random_state=SEED)

for name, df in splits.items():
    print(f"{name}: {len(df)} rows")

train: 2000 rows
validation: 300 rows
test: 1320 rows


# Build DPO JSONL file from dataset

In [15]:
# Même system prompt que pour le SFT : le DPO part du modèle SFT et doit voir le même gabarit
SYSTEM_PROMPT = "Tu es un assistant médical expert. Réponds de façon précise et concise."


def to_prompt(row) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{row['clinical_case']}\n\n{row['question']}"},
    ]


def to_assistant(text: str) -> list[dict]:
    return [{"role": "assistant", "content": text}]


METADATA = ["id", "source", "medical_subject", "question_type", "rejected_type", "pair_check", "golden_answer", "lang"]

# Un fichier JSONL par split : ../data/dpo_mediqal_mcqm_{train,validation,test}.jsonl
dpo_ds = DatasetDict()
for name, df in splits.items():
    df = df.assign(
        source="MediQAl-mcqm",
        pair_check="golden_answer",  # rejected construit à partir du corrigé officiel
        golden_answer=df["correct_answers"],
        lang="fr",
        prompt=df.apply(to_prompt, axis=1),
        chosen=df["chosen"].map(to_assistant),
        rejected=df["rejected"].map(to_assistant),
    )
    dpo_ds[name] = Dataset.from_pandas(df[METADATA + ["prompt", "chosen", "rejected"]], preserve_index=False)
    dpo_ds[name].to_json(f"../data/dpo_mediqal_mcqm_{name}.jsonl", force_ascii=False)

dpo_ds["train"][0]

Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 47.58ba/s]


{'id': '8187',
 'source': 'MediQAl-mcqm',
 'medical_subject': 'Nephro-Urology',
 'question_type': 'Understanding',
 'rejected_type': 'omission',
 'pair_check': 'golden_answer',
 'golden_answer': 'A,D',
 'lang': 'fr',
 'prompt': [{'role': 'system',
   'content': 'Tu es un assistant médical expert. Réponds de façon précise et concise.'},
  {'role': 'user',
   'content': "Un homme de 38 ans, célibataire, consulte pour une grosse bourse droite ; il rapporte que l'augmentation de volume a été progressive en deux à trois mois sans douleur et sans fièvre. L'examen clinique attentif permet de bien différencier l'épididyme et d'affirmer qu'il s'agit d'une tumeur testiculaire déformant le testicule droit qui est deux fois plus gros que le gauche. L'examen clinique général est normal\n\nQuel(s) est(sont) le(s) marqueur(s) biologique(s) le plus souvent rencontré(s) chez un sujet présentant un cancer du testicule ?"}],
 'chosen': [{'role': 'assistant',
   'content': "- Alphafoetoprotéine\n- Fractio

In [16]:
print(dpo_ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'medical_subject', 'question_type', 'rejected_type', 'pair_check', 'golden_answer', 'lang', 'prompt', 'chosen', 'rejected'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['id', 'source', 'medical_subject', 'question_type', 'rejected_type', 'pair_check', 'golden_answer', 'lang', 'prompt', 'chosen', 'rejected'],
        num_rows: 300
    })
    test: Dataset({
        features: ['id', 'source', 'medical_subject', 'question_type', 'rejected_type', 'pair_check', 'golden_answer', 'lang', 'prompt', 'chosen', 'rejected'],
        num_rows: 1320
    })
})


# Anonymisation des données

On n'anonymise que le **prompt** (cas clinique + question), qui décrit le patient. Les réponses `chosen` / `rejected` sont des options de QCM rédigées par les enseignants : des énoncés médicaux courts, sans donnée patient. Presidio y prend presque chaque terme en début de ligne pour un nom (« Angiographie », « Pyélonéphrite », « Érythrose palmaire »…) : anonymiser les options effacerait les éléments que la paire compare, et pourrait même rendre `chosen` et `rejected` identiques.

Contrôle de compensation : la cellule suivante liste les options qui contiennent une civilité suivie d'un nom, à relire à la main.

In [17]:
# Options contenant une civilité suivie d'un nom (« Monsieur X. », « Mme Dupont ») : à relire
TITLE_IN_OPTION = re.compile(r"\b(?:Monsieur|Madame|Mademoiselle|Mme|Mlle|M\.|Mr|Dr)\s+[A-ZÀ-Ý]")
flagged = [
    (name, row["id"], text)
    for name, df in splits.items()
    for _, row in df.iterrows()
    for text in (row["chosen"], row["rejected"])
    if TITLE_IN_OPTION.search(text)
]
print(f"{len(flagged)} réponses à relire")
pd.DataFrame(flagged, columns=["split", "id", "réponse"]).drop_duplicates().head(20)

9 réponses à relire


,split,id,réponse
0,train,17559,- Il existe une hépathopathie. Monsieur B. con...
1,train,17559,- Il existe une hépathopathie. Monsieur B. con...
2,train,6818,- Indemnités journalières au cours du congé pr...
3,train,12740,- La mère de Monsieur X. est asthmatique\n- Mo...
4,train,12740,- La mère de Monsieur X. est asthmatique\n- Mo...
5,train,5098,- Sérothérapie spécifique pour Mlle X...\n- Va...
6,train,5098,- Sérothérapie spécifique pour Mlle X...
7,test,13631,- Le certificat du médecin demandant le placem...
8,test,7184,- Vous vous retranchez derrière le secret prof...


In [18]:
# Anonymisation de chaque split : ../data/anonymized/dpo_mediqal_mcqm_{split}.anonymized.jsonl
reports = {
    name: run(f"../data/dpo_mediqal_mcqm_{name}.jsonl", lang="fr", output_dir="../data/anonymized", fields=["prompt"])
    for name in dpo_ds
}
pd.DataFrame(reports).T[["records", "records_modified", "persons_replaced"]]

,records,records_modified,persons_replaced
train,2000,675,1233
validation,300,96,174
test,1320,442,770


In [19]:
# DataFrame des champs "content" des jsonl anonymisés : une ligne par message (hors system)
rows = []
for name in dpo_ds:
    with open(f"../data/anonymized/dpo_mediqal_mcqm_{name}.anonymized.jsonl", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            for field in ["prompt", "chosen", "rejected"]:
                for message in record[field]:
                    if message["role"] != "system":
                        rows.append({"split": name, "id": record["id"], "field": field, "content": message["content"]})

contents_df = pd.DataFrame(rows)
contents_df["n_person"] = contents_df["content"].str.count("<PERSON>")
print(f"{len(contents_df)} messages, dont {(contents_df['n_person'] > 0).sum()} avec au moins une balise <PERSON>")
contents_df.groupby("field")["n_person"].agg(["sum", lambda s: (s > 0).sum()]).set_axis(["balises", "messages"], axis=1)

10860 messages, dont 1213 avec au moins une balise <PERSON>


,balises,messages
field,,
chosen,0,0
prompt,1773,1213
rejected,0,0


In [20]:
def person_contexts(text: str, width: int = 60) -> list[str]:
    """Extrait autour de chaque balise <PERSON>, avec `width` caractères de chaque côté."""
    return [text[max(0, m.start() - width) : m.end() + width].replace("\n", " ") for m in re.finditer("<PERSON>", text)]


# Une ligne par balise <PERSON>, pour vérifier les remplacements
occurrences_df = contents_df[["split", "id", "field"]].copy()
occurrences_df["context"] = contents_df["content"].apply(person_contexts)
occurrences_df = occurrences_df.explode("context").dropna(subset=["context"])

pd.set_option("display.max_colwidth", None)
print(f"{len(occurrences_df)} occurrences de <PERSON>")
occurrences_df.sample(min(20, len(occurrences_df)), random_state=SEED)

1773 occurrences de <PERSON>


,split,id,field,context
7287,test,16153,prompt,". Après un bref examen somatique qui s'avère normal, Madame <PERSON> peut raconter son histoire : il y a 2 ans, elle a perdu son"
7212,test,6317,prompt,"icromol/l (50-120 ; Calcémie : 2,55 millimol/l (2,24-2,6) ; <PERSON> : 589 micromol/l (150-450). Protides totaux : 80 g/lAlbumin"
4659,train,19041,prompt,"Monsieur <PERSON>.., 28 ans, est hospitalisé en urgence du fait de la survenu"
10146,test,14942,prompt,"Sur les radiographies, on constate une ostéocondensation de <PERSON> et de l'hémibassin droit, avec discrète protrusion acétabul"
7026,test,10794,prompt,"e des nerfs crâniens. L'acoumétrie aux diapasons montre un <PERSON> localisé à droite, un <PERSON> négatif des deux côtés. Vous"
6216,validation,1896,prompt,ct est posé et les métaux sont soupçonnés comme allergènes <PERSON>) complication(s) doit-on redouter ?
2949,train,20787,prompt,"oi de VRP. L'interne de la clinique vérifie l'adresse de Mr <PERSON>, le certificat du médecin de famille et demande à l'ambulan"
9135,test,13769,prompt,"Voici six mois, à table, <PERSON>..., 46 ans, à éprouvé le sentiment que ses enfants ne lui r"
6330,validation,18226,prompt,"e, est admise d'urgence à 36 semaines d'aménorrhée pour une <PERSON> à 18/12, avec une protéinurie à 1 gramme/litre. Elle mesure"
9699,test,10627,prompt,L'épreuve de Weber montre une absence de latéralisation. Le <PERSON> est négatif des deux côtés. Le diagnostic le plus probable


# Contrôle final

In [21]:
# Rechargement des fichiers anonymisés, tels que les lira le notebook d'entraînement DPO
DATA_FILES = {name: f"../data/anonymized/dpo_mediqal_mcqm_{name}.anonymized.jsonl" for name in dpo_ds}
final_ds = load_dataset("json", data_files=DATA_FILES)
print(final_ds)

for name, split in final_ds.items():
    same = sum(c[0]["content"] == r[0]["content"] for c, r in zip(split["chosen"], split["rejected"]))
    assert same == 0, f"{name}: {same} paires avec chosen == rejected"
print("OK : aucune paire avec chosen == rejected")

Generating train split: 2000 examples [00:00, 92359.10 examples/s]
Generating validation split: 300 examples [00:00, 22146.00 examples/s]
Generating test split: 1320 examples [00:00, 89446.68 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'medical_subject', 'question_type', 'rejected_type', 'pair_check', 'golden_answer', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['id', 'source', 'medical_subject', 'question_type', 'rejected_type', 'pair_check', 'golden_answer', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 300
    })
    test: Dataset({
        features: ['id', 'source', 'medical_subject', 'question_type', 'rejected_type', 'pair_check', 'golden_answer', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 1320
    })
})
OK : aucune paire avec chosen == rejected


# Limites

- **Options absentes du prompt** : comme pour le SFT, le modèle reçoit le cas et la question, mais pas la liste des options. Certaines questions (« Parmi les propositions suivantes… ») y font référence : le modèle doit produire les bonnes réponses sans les voir.
- **Réponses courtes, sans justification** : les paires apprennent *quoi* répondre, pas à *expliquer* la réponse. Les explications viennent de la partie anglaise (UltraMedical).
- **Erreurs simulées** : les `rejected` sont construits, pas produits par le modèle. Étape suivante possible : faire répondre le modèle SFT à ces questions et prendre ses réponses fausses (vérifiables par le corrigé) comme `rejected`.